In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from tqdm.notebook import tqdm

In [2]:
def extract_tags(soup):
    try:
        tags = [tag for tag in soup.find_all("a") if "class" in tag.attrs and "app_tag" in tag["class"]]
        best_app_tags = [tag.text.strip() for tag in tags][:5]
        return best_app_tags
    except:
        return []

def extract_rating(soup):
    try:
        rating_spans = soup.find_all("span", class_="game_review_summary")
        for rating in rating_spans:
            if "data-tooltip-html" in rating.attrs:
                rating_text_arr = rating["data-tooltip-html"].split()

                # find the overall rating, not the recent rating
                if "30" not in rating_text_arr:
                    return int(rating_text_arr[0].replace("%", ""))
    except:
        return None

def get_soup(game_id):
    url = f"https://store.steampowered.com/app/{game_id}"
    response = requests.get(url)
    return BeautifulSoup(response.text, "html.parser")

def get_game_data(game_id):
    soup = get_soup(game_id)
    tags = extract_tags(soup)
    rating = extract_rating(soup)
    return tags, rating


In [6]:
game_folder = "../data/game_data.csv"
time_folder = "../data/time_data.csv"

game_data = pd.read_csv(game_folder)
time_data = pd.read_csv(time_folder)

In [7]:
len(game_data)

31767

In [8]:
game_time_sum = time_data.groupby('game_id')['user_id'].count()
game_time_sum = game_time_sum.reset_index()
game_time_sum.columns = ['game_id', 'users']
game_time_sum = game_time_sum.sort_values(by='users', ascending=False)
# get ids of games with at least 5 users, and filter the time_data
game_time_sum = game_time_sum[game_time_sum['users'] >= 5]

# get the game ids
game_ids = game_time_sum['game_id']

# filter the game_data
game_data = game_data[game_data['game_id'].isin(game_ids)]
len(game_data)

20204

In [ ]:
for i, row in tqdm(game_data.iterrows(), total=len(game_data)):
    game_id = row["game_steam_id"]
    tags, rating = get_game_data(game_id)
    for j in range(5):
        if j < len(tags):
            game_data.at[i, f"tag_{j}"] = tags[j]
    game_data.at[i, "rating"] = rating

In [24]:
game_data.head(100)

,game_id,game_steam_id,game_name,tag_0,tag_1,tag_2,tag_3,tag_4,rating
0,1,4000,Garry's Mod,Sandbox,Moddable,Multiplayer,Physics,Building,96.0
1,2,400,Portal,Puzzle,Puzzle Platformer,First-Person,3D Platformer,Singleplayer,98.0
2,3,20900,The Witcher: Enhanced Edition,RPG,Fantasy,Story Rich,Mature,Singleplayer,89.0
3,4,22000,World of Goo,Puzzle,Indie,Physics,Singleplayer,Great Soundtrack,94.0
4,5,17410,Mirror's Edge,Action-Adventure,Parkour,First-Person,Exploration,3D Platformer,87.0
...,...,...,...,...,...,...,...,...,...
95,96,271570,Space Farmers,Indie,Action,Co-op,Space,Multiplayer,78.0
96,97,271670,10 Second Ninja,Indie,Action,Platformer,Ninja,Fast-Paced,90.0
97,98,278360,A Story About My Uncle,Adventure,First-Person,Parkour,Singleplayer,Indie,92.0
98,99,280640,Dark Shadows - Army of Evil,Indie,Action,RPG,First-Person,Fantasy,33.0


In [25]:
# save as a csv
game_data.to_csv("../data/game_data_with_tags.csv", index=False)